[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/45_ntk_rope_solution.ipynb)

# 🟡 Solution: NTK-aware RoPE Scaling

Reference solution for NTK-aware Rotary Position Embedding scaling.

$$\text{base\_new} = 10000 \cdot \text{scale}^{D/(D-2)}$$

Then apply standard RoPE with the new base.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def ntk_rope(q, k, scale):
    B, S, D = q.shape
    new_base = 10000.0 * (scale ** (D / (D - 2)))
    pos = torch.arange(S, device=q.device).float().unsqueeze(1)
    dim = torch.arange(0, D, 2, device=q.device).float()
    freqs = 1.0 / (new_base ** (dim / D))
    angles = pos * freqs
    cos_a = torch.cos(angles)
    sin_a = torch.sin(angles)

    def rotate(x):
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos_a - x2 * sin_a,
                            x1 * sin_a + x2 * cos_a], dim=-1).flatten(-2)

    return rotate(q), rotate(k)

In [ ]:
# Verify: compare scale=1 vs scale=4 frequencies, show norm preservation
B, S, D = 1, 8, 16
q = torch.randn(B, S, D)
k = torch.randn(B, S, D)

q1, k1 = ntk_rope(q, k, scale=1.0)
q4, k4 = ntk_rope(q, k, scale=4.0)

print("scale=1 base:", 10000.0 * (1.0 ** (D / (D - 2))))
print("scale=4 base:", 10000.0 * (4.0 ** (D / (D - 2))))
print()
print("Norm preservation (scale=1):")
print("  q input norm:", q.norm(dim=-1).mean().item())
print("  q output norm:", q1.norm(dim=-1).mean().item())
print()
print("Norm preservation (scale=4):")
print("  q input norm:", q.norm(dim=-1).mean().item())
print("  q output norm:", q4.norm(dim=-1).mean().item())

In [ ]:
from torch_judge import check
check("ntk_rope")